In [1]:
import requests
import io
import csv


def test_github_taxonomy():
    url = "https://raw.githubusercontent.com/datasets/harmonized-system/master/data/harmonized-system.csv"
    print(f"Fetching sample data from: {url}\n")

    response = requests.get(url)
    if response.status_code != 200:
        print(f"Error fetching data. Status: {response.status_code}")
        return

    csv_file = io.StringIO(response.text)
    reader = csv.DictReader(csv_file)

    print("--- SAMPLE MATCHES FOUND IN SCOPE ---")
    count = 0

    for row in reader:
        code = row.get('hscode', '').strip()
        description = row.get('description', '').strip()

        # Look for 6-digit codes matching our hackathon scope:
        # Chapter 30 (Pharma), Chapter 90 (Medical Instruments), Chapter 93 (Ammunition)
        if len(code) == 6 and (code.startswith('30') or code.startswith('9018') or code.startswith('9022') or code.startswith('9306')):
            print(
                f"Code: {code} | Level: {row.get('level')} | Parent: {row.get('parent')}")
            print(f"Description: {description}")
            print("-" * 50)
            count += 1

        if count >= 5:  
            break

In [2]:
test_github_taxonomy()

Fetching sample data from: https://raw.githubusercontent.com/datasets/harmonized-system/master/data/harmonized-system.csv

--- SAMPLE MATCHES FOUND IN SCOPE ---
Code: 300120 | Level: 6 | Parent: 3001
Description: Glands and other organs; extracts of glands or other organs or of their secretions, for organo-therapeutic uses
--------------------------------------------------
Code: 300190 | Level: 6 | Parent: 3001
Description: Glands and other organs; heparin and its salts; other human or animal substances prepared for therapeutic or prophylactic uses, n.e.c. in heading 3001
--------------------------------------------------
Code: 300212 | Level: 6 | Parent: 3002
Description: Blood, human or animal, antisera, other blood fractions and immunological products; antisera and other blood fractions
--------------------------------------------------
Code: 300213 | Level: 6 | Parent: 3002
Description: Blood, human or animal, antisera, other blood fractions and immunological products; immunologica

In [7]:
import requests
import json


def test_wits_nonzero_tariff():
    # Reporter: 918 (European Union - Corrected Numeric Code)
    # Partner: 000 (World/MFN)
    # Product: 930630 (Other cartridges and parts thereof - Ammunition)
    url = "https://wits.worldbank.org/API/V1/SDMX/V21/datasource/TRN/reporter/918/partner/000/product/930630/year/2022/datatype/reported?format=JSON"

    headers = {
        "Accept": "application/json",
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
    }

    print(f"Executing GET request to:\n{url}\n")

    try:
        response = requests.get(url, headers=headers)

        if response.status_code == 200:
            data = response.json()

            # Drill straight down to the observation array
            obs_array = data["dataSets"][0]["series"]["0:0:0:0:0"]["observations"]["0"]

            print("Response Payload Caught!")
            print(f"Raw SDMX Array: {obs_array}\n")

            print("--- PARSED DATA ---")
            print(f"Ad Valorem Rate (Position 0): {obs_array[0]}%")
            print(f"Tariff Type (Position 1):     Index {obs_array[1]} (MFN)")
            print(f"Total Tariff Lines:           {obs_array[3]}")
            print(f"Specific Duty Lines count:    {obs_array[5]}")

        else:
            print(f"Error Response: {response.status_code}")
            print(response.text)

    except Exception as e:
        print(f"Request failed: {e}")


if __name__ == "__main__":
    test_wits_nonzero_tariff()

Executing GET request to:
https://wits.worldbank.org/API/V1/SDMX/V21/datasource/TRN/reporter/918/partner/000/product/930630/year/2022/datatype/reported?format=JSON

Response Payload Caught!
Raw SDMX Array: [2.36666655540466, 0, None, 0, 0, 0, 0, 0, 0, 0, 0, 0]

--- PARSED DATA ---
Ad Valorem Rate (Position 0): 2.36666655540466%
Tariff Type (Position 1):     Index 0 (MFN)
Total Tariff Lines:           0
Specific Duty Lines count:    0
